In [3]:
import pandas as pd

df = pd.read_csv("Car-Data-Occasion.csv")  # adapte le chemin si besoin

# 1) Supprimer les lignes où le prix est 0
df = df[df["Prix"] != 0]

# 2) Supprimer les doublons 
df = df.drop_duplicates()

#  Sauvegarder dans un nouveau fichier
df.to_csv("Car-Data-Occasion-Clean-1.csv", index=False)


In [4]:
df = pd.read_csv("Car-Data-Occasion-Clean-1.csv")

# 1) Supprimer la colonne 'Nombre Portes'
df = df.drop(columns=["Nombre Portes"])

# 2) Corriger les prix
# - Prix entre 10 et 100 -> * 1000
mask_10_100 = (df["Prix"] <= 100)
df.loc[mask_10_100, "Prix"] = df.loc[mask_10_100, "Prix"] * 1000

# Sauvegarde
df.to_csv("Car-Data-Occasion-Clean-2.csv", index=False)


In [9]:
import pandas as pd


df = pd.read_csv("Car-Data-Occasion-Clean-2.csv")

# 1) Calculer la moyenne des prix par (Marque, Modele)
df["Prix_moyen_modele"] = df.groupby(["Marque", "Modele"])["Prix"].transform("mean")

# 2) Définir les prix illogiques
mask_illogique = (df["Prix"] < 5000) | (df["Prix"] > 1000000)

# 3) Remplacer par la moyenne du groupe
df.loc[mask_illogique, "Prix"] = df.loc[mask_illogique, "Prix_moyen_modele"]

# 4) (Optionnel) Si certains groupes n'ont qu'une seule ligne et un prix illogique,
# la moyenne est identique; tu peux décider de les supprimer:
# df = df[~mask_illogique | df["Prix"].notna()]

# 5) Supprimer la colonne temporaire
df = df.drop(columns=["Prix_moyen_modele"])

df["Prix"] = df["Prix"].round().astype(int)
# Sauvegarde
df.to_csv("Car-Data-Occasion-Clean-3.csv", index=False)


In [10]:
import pandas as pd
import numpy as np

df = pd.read_csv("Car-Data-Occasion-Clean-3.csv")

# 1) Nettoyer la colonne Kilometrage (enlever espaces, gérer N/A)
df["Kilometrage"] = (
    df["Kilometrage"]
    .astype(str)
    .str.replace(" ", "", regex=False)   # "181 000" -> "181000"
    .replace("N/A", np.nan)             # N/A -> NaN
)

df["Kilometrage"] = pd.to_numeric(df["Kilometrage"])

# 2) Définir les valeurs invalides: manquantes, <= 0, ou > 999999
mask_invalide = df["Kilometrage"].isna() | (df["Kilometrage"] <= 0) | (df["Kilometrage"] > 999_999)

# 3) Calculer la moyenne du Kilometrage (sur les valeurs valides uniquement)
moyenne_km = df.loc[~mask_invalide, "Kilometrage"].mean()

# 4) Remplacer les valeurs invalides par la moyenne
df.loc[mask_invalide, "Kilometrage"] = moyenne_km

# 5) arrondir et convertir en entier
df["Kilometrage"] = df["Kilometrage"].round().astype(int)

# Sauvegarde
df.to_csv("Car-Data-Occasion-Clean-4.csv", index=False)


In [11]:
import pandas as pd

df = pd.read_csv("Car-Data-Occasion-Clean-4.csv")

cols_cat = ["Carburant", "Boite Vitesse", "Puissance Fiscale"]

for col in cols_cat:
    # 1) Valeur la plus fréquente par (Marque, Modele)
    mode_by_model = (
        df.groupby(["Marque", "Modele"])[col]
          .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else pd.NA)
    )

    # 2) Mapper sur le dataframe
    df[col + "_mode_model"] = df.set_index(["Marque", "Modele"]).index.map(mode_by_model)

    # 3) Remplir d’abord avec le mode du modèle
    df[col] = df[col].fillna(df[col + "_mode_model"])

    # 4) Puis avec le mode global si encore NaN
    global_mode = df[col].mode().iloc[0]
    df[col] = df[col].fillna(global_mode)

    # 5) Supprimer la colonne temporaire
    df.drop(columns=[col + "_mode_model"], inplace=True)

# Sauvegarde
df.to_csv("Car-Data-Occasion-Clean-5.csv", index=False)
